# IT594 Project: Notebook 02, Amazon ESCI exploration (secondary dataset)

**Why this notebook exists:** WANDS has only about 105 constrained queries out of 480. ESCI is far larger,
so we use it to check whether the findings from WANDS still hold on a second dataset.

**Questions this notebook answers**
1. Is ESCI loadable, and how big is the English (`us`) part?
2. How many queries name a color, size, or material?
3. What are the `negations` queries (explicit "without / not" requirements)?
4. Is `product_color` filled in often enough to build a color-constraint check?
5. Do ESCI labels (Exact / Substitute / Complement / Irrelevant) also contain color violations?

**Before running:** download the ESCI parquet files. The repo stores them with Git LFS, so a plain
`git clone` gives tiny pointer files. Use `git lfs pull` or the official download page.

## 1. Setup

In [1]:
import sys
print(sys.executable)

d:\constraint-search\venv\Scripts\python.exe


In [2]:
import os, re
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 100)

PATH = "../data/esci-data/shopping_queries_dataset/"   # change if needed
EX_FILE  = PATH + "shopping_queries_dataset_examples.parquet"
PR_FILE  = PATH + "shopping_queries_dataset_products.parquet"
SRC_FILE = PATH + "shopping_queries_dataset_sources.csv"

# Detect Git LFS pointer files (tiny text files instead of real data)
for f in [EX_FILE, PR_FILE]:
    size = os.path.getsize(f)
    print(f"{os.path.basename(f):55s} {size/1e6:10.1f} MB")
    assert size > 10_000, (
        f"{f} is only {size} bytes. It is a Git LFS pointer, not the real data. "
        "Run `git lfs pull` inside the esci-data folder."
    )

USE_SMALL = False   # True = reduced 'easy-filtered' version (48,300 queries); False = large (130,652)

shopping_queries_dataset_examples.parquet                     51.3 MB
shopping_queries_dataset_products.parquet                   1108.9 MB


## 2. Load examples (English only)
ESCI is multilingual (`us`, `es`, `jp`). We keep `us` so that color and size words are English.

In [3]:
ex = pd.read_parquet(EX_FILE, columns=["example_id", "query", "query_id", "product_id",
                                      "product_locale", "esci_label", "small_version", "large_version"])
ver = "small_version" if USE_SMALL else "large_version"
ex = ex[(ex["product_locale"] == "us") & (ex[ver] == 1)].copy()

print("rows:", len(ex), "| unique queries:", ex["query_id"].nunique(),
      "| unique products:", ex["product_id"].nunique())
print()
print(ex["esci_label"].value_counts())
print()
print("Judged products per query (min / median / max):",
      ex.groupby("query_id").size().agg(["min", "median", "max"]).tolist())
ex.head()

rows: 1818825 | unique queries: 97345 | unique products: 1215851

esci_label
E    1247558
S     369313
I     161905
C      40049
Name: count, dtype: int64

Judged products per query (min / median / max): [8.0, 16.0, 198.0]


,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1
1,1,revent 80 cfm,0,B07X3Y6B1V,us,E,0,1
2,2,revent 80 cfm,0,B07WDM7MQQ,us,E,0,1
3,3,revent 80 cfm,0,B07RH6Z8KW,us,E,0,1
4,4,revent 80 cfm,0,B07QJ7WYFQ,us,E,0,1


Label meanings: **E** Exact, **S** Substitute (functional but not what was asked),
**C** Complement (goes with it), **I** Irrelevant.

## 3. Attach the query `source` tag
The sources file says how each query was collected (`negations`, `parse_pattern`, `behavioral`, `other`, `nlqec`).

In [4]:
src = pd.read_csv(SRC_FILE)
ex = ex.merge(src, on="query_id", how="left")

qdf = ex[["query_id", "query", "source"]].drop_duplicates("query_id")
print(qdf["source"].value_counts())

source
other            88774
parse_pattern     4250
negations         2899
behavioral        1253
nlqec              169
Name: count, dtype: int64


## 4. How many queries name a candidate constraint?
Simple regex scan, a rough first pass (same idea as in notebook 01).

In [5]:
COLORS = ["black","white","blue","navy","gray","grey","red","green","beige","brown","pink","yellow",
          "gold","silver","ivory","cream","purple","orange","teal","turquoise"]
MATERIALS = ["wood","wooden","leather","metal","glass","marble","velvet","fabric","linen","rattan",
             "wicker","bamboo","steel","iron","brass","cotton","wool","acrylic","ceramic","plastic",
             "stainless","silicone"]
NEG_WORDS = ["without", "no", "not", "non", "free", "except", "excluding"]

def has_word(q, words):
    toks = set(re.findall(r"[a-z]+", q.lower()))
    return any(w in toks for w in words)

SIZE_PATTERN = re.compile(r"\b\d+(\.\d+)?\s*(x|\')\s*\d+|\b\d+(\.\d+)?\s*(ft|foot|feet|inch|inches|in|cm|mm|oz|lb|lbs|gallon|ml|l)\b", re.I)

qdf["has_color"]    = qdf["query"].apply(lambda q: has_word(q, COLORS))
qdf["has_material"] = qdf["query"].apply(lambda q: has_word(q, MATERIALS))
qdf["has_size"]     = qdf["query"].apply(lambda q: bool(SIZE_PATTERN.search(q)))
qdf["has_negation"] = qdf["query"].apply(lambda q: has_word(q, NEG_WORDS))
qdf["any_constraint"] = qdf[["has_color", "has_material", "has_size"]].any(axis=1)

summary = qdf[["has_color", "has_material", "has_size", "has_negation", "any_constraint"]].agg(["sum", "mean"]).T
summary.columns = ["n_queries", "share"]
summary["share"] = (summary["share"] * 100).round(1)
print("Total queries:", len(qdf))
summary

Total queries: 97345


,n_queries,share
has_color,6226.0,6.4
has_material,3115.0,3.2
has_size,2802.0,2.9
has_negation,3890.0,4.0
any_constraint,11536.0,11.9


In [6]:
# Does the source tag predict constraints?
qdf.groupby("source")[["has_color", "has_material", "has_size", "has_negation"]].agg(["sum"]).droplevel(1, axis=1) \
   .assign(n_queries=qdf.groupby("source").size())

,has_color,has_material,has_size,has_negation,n_queries
source,,,,,
behavioral,47,36,14,21,1253
negations,88,146,199,2899,2899
nlqec,43,15,4,14,169
other,5326,2771,2390,908,88774
parse_pattern,722,147,195,48,4250


## 5. Read the `negations` queries
These look like explicit exclusions ("without ...", "not ..."). **Read them yourself** before relying on
that interpretation. The README does not define the tags.

In [7]:
neg = qdf[qdf["source"] == "negations"]
print(len(neg), "negation-tagged queries")
neg["query"].sample(min(25, len(neg)), random_state=1).tolist()

2899 negation-tagged queries


['organic multivitamin without biotin',
 'crusader without violence',
 'discipline without punishment',
 '- power auto folding mirrors corner lights not painted',
 'the importance of not being',
 'evangelion 1.11 you are not alone',
 'cigarettes without tobacco',
 'webcam without microphone',
 'swim tank without bra',
 '11.5x17.2 stainless steel grill without handles',
 'stop im not yours luggage tag',
 '1 3/8 inch bathroom drain stopper without overflow',
 'just a mom trying not to raise liberals tshirt',
 '*smoke brisket not meth* shirt',
 '100% success without college',
 'ring light without stand 18',
 'phytoceramides without wheat',
 'menopause relief without soy',
 'marriage without regrets',
 'pc headphones without microphone',
 'even without you',
 'puzzles without pictures',
 '100 ct 1ml syringe without needle',
 'shakespeare not stirred',
 'plastic mailboxes for outside without post']

In [8]:
pp = qdf[qdf["source"] == "parse_pattern"]
print(len(pp), "parse_pattern-tagged queries")
pp["query"].sample(min(25, len(pp)), random_state=1).tolist()

4250 parse_pattern-tagged queries


['gig bag acoustic guitar',
 'gift for 2 year old boy',
 '2x womens v neck tshirts',
 'wall canvas black and white',
 'toddler alarm clock red green',
 'copper glasses water',
 'crest prohealth toothpaste',
 'floor vases decorative tall 36 inches',
 'orange cream meal replacement drink',
 'porsche motorsport',
 'tsa zip lock',
 'blue green wallpaper',
 'womens rugby shirts long sleeve',
 'hi mirror mini premium x',
 '3 bead necklace pearl',
 'precise tall womens golf m5',
 'neon apple watch band 44mm',
 'brown leather patches for furniture',
 'low power am transmitter',
 'leather messenger bags women',
 'hospital respirator mask',
 'wireless horn with remote',
 'baby shower mint molds',
 'aquateak the original 18" asia teak shower bench',
 'garmin satellite phone']

## 6. Product metadata: is `product_color` usable?
The products file is large, so we load only the columns we need and only English rows.

In [9]:
prod = pd.read_parquet(PR_FILE,
                       columns=["product_id", "product_title", "product_brand", "product_color", "product_locale"],
                       filters=[("product_locale", "==", "us")])
prod = prod[prod["product_id"].isin(ex["product_id"].unique())]

print("products used:", len(prod))
has_color = prod["product_color"].notna() & (prod["product_color"].astype(str).str.strip() != "")
print(f"product_color filled: {has_color.mean():.1%}")
print()
print(prod.loc[has_color, "product_color"].str.lower().value_counts().head(25))

products used: 1215851
product_color filled: 66.6%

product_color
black              132763
white               58120
blue                24176
silver              18540
red                 17681
multicolor          15353
green               12235
clear               12069
brown               11948
pink                11754
grey                11487
gray                 9303
multi                7545
yellow               7384
gold                 6806
purple               5885
navy                 5215
orange               4784
beige                3687
stainless steel      3202
rose gold            2716
natural              2701
multicolored         2490
black/white          2221
multi-colored        2185
Name: count, dtype: int64


Note the messy values (multi-word colors, brand-like strings, mixed languages). Same ambiguity problem as WANDS.

## 7. Do ESCI labels contain color violations?
Same test as WANDS: for queries that name a color, compare the label with the product's color field.

In [10]:
color_q = qdf[qdf["has_color"]][["query_id", "query"]].copy()
color_q["qcolors"] = color_q["query"].apply(
    lambda q: [c for c in COLORS if c in set(re.findall(r"[a-z]+", q.lower()))])

m = (ex.merge(color_q, on=["query_id", "query"])
       .merge(prod[["product_id", "product_color"]], on="product_id", how="left"))

def check(row):
    c = row["product_color"]
    if pd.isna(c) or str(c).strip() == "":
        return "missing"
    c = str(c).lower()
    return "match" if any(q in c for q in row["qcolors"]) else "mismatch"

m["color_check"] = m.apply(check, axis=1)
print("color-query pairs:", len(m), "| queries:", m["query_id"].nunique())
print()
print(pd.crosstab(m["esci_label"], m["color_check"]))
print()
print(pd.crosstab(m["esci_label"], m["color_check"], normalize="index").round(3))

color-query pairs: 121776 | queries: 6226

color_check  match  mismatch  missing
esci_label                           
C              749       654      387
E            46582     16068    11871
I             3268      4030     2294
S            12436     16392     7045

color_check  match  mismatch  missing
esci_label                           
C            0.418     0.365    0.216
E            0.625     0.216    0.159
I            0.341     0.420    0.239
S            0.347     0.457    0.196


**How to read it:** among `E` (Exact) rows, any `mismatch` is a labeled-relevant product that violates the
queried color. Compare the mismatch share for `E` with the share for `I`. If `E` rows still mismatch often,
the benchmark labels themselves contain constraint violations (same finding as in WANDS?).

In [11]:
# Concrete examples: Exact label but the color does not match
bad = m[(m["esci_label"] == "E") & (m["color_check"] == "mismatch")] \
        .merge(prod[["product_id", "product_title"]], on="product_id", how="left")
bad[["query", "product_title", "product_color"]].sample(min(15, len(bad)), random_state=1)

,query,product_title,product_color
12449,round blue light glasses,"FEIYOLD Blue Light Blocking Glasses Women/Men,Retro Round Anti Eyestrain Computer Gaming Glasses...",Leopard+crystal
205,1.63 black drain without overflow,Phiestina Commercial Solid Brass Pup Up Matte Black Lavatory Bathroom Sink Drain Assembly With B...,Matte Balck With Overflow
7138,green advertine crystal,Lifestyle on Fire Healing Crystals & Chakra Stones Set - 12 Piece Set with Crystals and Healing ...,Multicolor
6508,gold link bracelets,Gold Chain Bracelet Sets for Women Girls 14K Gold Plated Dainty Link Paperclip Choker Bracelet S...,Style-2
14465,wall canvas black and white,Bathroom Decor Canvas Wall Art Framed Black and White Wall Art Floral Wall Decor Print White Flo...,01-tulip
14950,white high waisted pants,"GLORIA VANDERBILT Women's Amanda Classic High Rise Tapered Jean, Black, 14 Long",Black
15435,womans pink robe,Just Love 6341-Fuchsia-M Kimono Robe/Hooded Bath Robes for Women,Fuchsia With Hood
6008,frosted glass white board,"Iceberg 31150 Clarity Glass Dry Erase White Board, 60"" x 36""",Gray
11877,red hangers,"Amazon Basics Slim, Velvet, Non-Slip Suit Clothes Hangers, Black/Silver - Pack of 100",Black/Silver
1253,bamboo earrings hoops gold,5 Pairs Gold Plated Geometric Statement Bamboo Hoop Loop Earrings for Women Girls Set.,6


## 8. Label to gain mapping (for NDCG later)
ESCI has four labels, so state your mapping explicitly in the report.

In [12]:
ESCI_GAIN = {"E": 3, "S": 2, "C": 1, "I": 0}     # one common choice; report any alternative as an ablation
ex["gain"] = ex["esci_label"].map(ESCI_GAIN)
print("Unmapped labels:", ex["gain"].isna().sum(), "(should be 0)")
qrels = {q: dict(zip(g["product_id"], g["gain"])) for q, g in ex.groupby("query_id")}
print("queries with qrels:", len(qrels))

Unmapped labels: 0 (should be 0)
queries with qrels: 97345


## 9. Findings (fill in after running)

**English queries (us), large version:**
- 1,818,825 rows, 97,345 queries, 1,215,851 products
- Labels: E 1,247,558 (68.6%), S 369,313 (20.3%), I 161,905 (8.9%), C 40,049 (2.2%)
- Judged products per query: min 8, median 16, max 198

**Queries naming color / size / material / negation:**
- color:6,226 (6.4%)
- size:2,802 (2.9%)
- material:3,115 (3.2%)
- negation:3,890 (4.0%)
- Any of color, material, size: 11,536 (11.9%)
- Compared with WANDS (about 105 queries), this is a large sample.

**`product_color` filled in:**
- 66.6% of products
- Values are noisy: Style-2, 6, 01-tulip, Matte Balck With Overflow, Leopard+crystal

**What do `negations` and `parse_pattern` queries look like?**
- negations (2,899 queries) all contain a word like "without" or "not". In my reading of the 25-query sample, about 14 are real product exclusions ("webcam without microphone", "ring light without stand", "syringe without needle", "puzzles without pictures"). The other 11 are book, song, or T-shirt titles ("discipline without punishment", "even without you").
- parse_pattern (4,250 queries) is full of attributes: "toddler alarm clock red green", "floor vases decorative tall 36 inches", "neon apple watch band 44mm".
- parse_pattern has about 3x more color queries than the rest: 722 of 4,250 (17.0%) against 5,326 of 88,774 (6.0%) in other.

**Exact-labeled color mismatches (count and share):**
- 121,776 color-query pairs across 6,226 queries
- Exact (E) pairs: 46,582 match, 16,068 mismatch, 11,871 missing
- That is 21.6% mismatch among all E pairs, or 25.6% among E pairs where color can be checked
Mismatch by label: S 45.7%, I 42.0%, C 36.5%, E 21.6%
- This is an upper bound. In the 15 examples you printed, about 4 were real violations (white pants that are Black, pink leggings that are Camouflage, red hangers that are Black/Silver, a white frame that is Black). About 4 were ambiguous (Dark Grey vs black, Multicolor, "white gold"). About 7 were checker or metadata errors: "blue light" is a feature, not a color; Violet for "purple"; Fuchsia for "pink"; and Style-2, 6, and 01-tulip aren't colors. One had a typo, "Balck".
- One example shows a violation the color check missed. "1.63 black drain without overflow" got an Exact label for a product whose color field says "With Overflow".

**Same finding as WANDS? Where it differs:**
- Same: Exact-labeled products still violate the stated color. WANDS had 17% mismatch, ESCI has about 22%. The two checkers differ, so don't compare the numbers too tightly.
- Different:
ESCI is about 60x larger, so results will be much more reliable.
Color is missing less often: 16% of E pairs, against 53% in WANDS.
Substitute (S) has the highest mismatch of any label (45.7%). ESCI's S label seems to absorb color-violating products, so it could work as a natural "related but violates a requirement" bucket.
- The same color problems appear in both: shade synonyms, bad field values, and multi-color products.

**Decision: use ESCI as a replication set? Which constraint types, how many queries?**
- Use ESCI as a replication set, with color as the main constraint (6,226 queries, 121,776 pairs).
- Add negation queries as a separate extension, but only the real product exclusions (hand-filter them first, since roughly half are titles).
- Treat size (2,802) and material (3,115) as optional.
- Before reporting any violation rate, fix the color checker:
Treat grey as gray, violet as purple, and fuchsia as pink.
Handle "white gold" and "blue light" as non-color phrases.
Ignore product_color values that aren't colors, and fall back to the title.
Hand-label about 200 pairs to measure the checker's precision and recall.

the checker and unit tests

In [13]:
import sys
sys.path.insert(0, "../src")
import color_checker
from color_checker import query_colors, color_check
print(color_checker.__doc__.splitlines()[0])

# ---- Unit tests: the real examples from your 15-row printout ----
tests = [  # (query, product_color, title, expected_in_strict_mode)
    ("round blue light glasses",          "Leopard+crystal",   "Blue Light Blocking Glasses", None),
    ("women white gold wedding bands",    "Silver",            "Titanium Eternity Ring",      None),
    ("frosted glass white board",         "Gray",              "Glass Dry Erase White Board", None),
    ("womans pink robe",                  "Fuchsia With Hood", "Kimono Robe",                 "match"),
    ("floral applique purple biolet",     "Color A Violet",    "Iron on Applique Patches",    "match"),
    ("1.63 black drain without overflow", "Matte Balck With Overflow", "Brass Drain",         "match"),
    ("gold link bracelets",               "Style-2",           "Gold Chain Bracelet Sets",    "match"),
    ("bamboo earrings hoops gold",        "6",                 "Gold Plated Hoop Earrings",   "match"),
    ("wall canvas black and white",       "01-tulip",          "Black and White Wall Art",    "match"),
    ("white high waisted pants",          "Black",             "Women's Jean, Black, 14",     "mismatch"),
    ("red hangers",                       "Black/Silver",      "Velvet Hangers",              "mismatch"),
    ("11x13 white frame without mat",     "Black",             "Picture Frame",               "mismatch"),
    ("womens pink capri leggings",        "Camouflage",        "Yoga Legging",                "unknown"),
    ("green advertine crystal",           "Multicolor",        "Crystal Set",                 "ambiguous"),
    ("long black parka jacket",           "Dark Grey",         "Ski Jacket",                  "mismatch"),
]
for q, pc, title, expected in tests:
    got = color_check(query_colors(q, False), pc, title, False)
    assert got == expected, f"{q!r} / {pc!r}: expected {expected}, got {got}"
print("All", len(tests), "unit tests passed.")

color_checker.py  (v2.1)  Color-constraint checker shared by all notebooks.
All 15 unit tests passed.


apply it and compare strict vs lenient

In [14]:
# Rebuild the color-query pairs with the NEW query extractor
qdf["qcols_strict"] = qdf["query"].apply(lambda q: query_colors(q, False))
color_ids = qdf.loc[qdf["qcols_strict"].apply(len) > 0, "query_id"]

m2 = (ex[ex["query_id"].isin(color_ids)][["query_id", "query", "product_id", "esci_label"]]
        .merge(prod[["product_id", "product_title", "product_color"]], on="product_id", how="left"))
m2["product_color"] = m2["product_color"].fillna("")
m2["product_title"] = m2["product_title"].fillna("")

def run(lenient):
    return [color_check(query_colors(q, lenient), c, t, lenient)
            for q, c, t in zip(m2["query"], m2["product_color"], m2["product_title"])]

m2["chk_strict"]  = run(False)
m2["chk_lenient"] = run(True)
print("color-query pairs:", len(m2), "| queries:", m2["query_id"].nunique())

def summarize(col):
    d = m2[m2[col].notna()]
    ct = pd.crosstab(d["esci_label"], d[col])
    for c in ["match", "mismatch", "ambiguous", "unknown"]:
        if c not in ct:
            ct[c] = 0
    ct = ct[["match", "mismatch", "ambiguous", "unknown"]]
    total = ct.sum(axis=1)
    ct["violation_rate_%"] = (100 * ct["mismatch"] / (ct["match"] + ct["mismatch"])).round(1)
    ct["unresolved_%"]     = (100 * (ct["ambiguous"] + ct["unknown"]) / total).round(1)
    return ct

print("\nSTRICT mode");  display(summarize("chk_strict"))
print("LENIENT mode"); display(summarize("chk_lenient"))
print("Old checker reported 21.6% mismatch for E (of all E pairs) and 45.7% for S.")

color-query pairs: 115603 | queries: 5960

STRICT mode


chk_strict,match,mismatch,ambiguous,unknown,violation_rate_%,unresolved_%
esci_label,,,,,,
C,966,391,39,282,28.8,19.1
E,56255,8316,727,5219,12.9,8.4
I,4031,2775,267,1984,40.8,24.9
S,15303,11872,901,6275,43.7,20.9


LENIENT mode


chk_lenient,match,mismatch,ambiguous,unknown,violation_rate_%,unresolved_%
esci_label,,,,,,
C,988,369,39,282,27.2,19.1
E,57426,7147,725,5219,11.1,8.4
I,4189,2617,267,1984,38.5,24.9
S,16113,11063,900,6275,40.7,20.9


Old checker reported 21.6% mismatch for E (of all E pairs) and 45.7% for S.


export the hand-labeling sample

In [19]:
import os
os.makedirs("annotations", exist_ok=True)

parts = []
for outcome, n in [("mismatch", 80), ("match", 80), ("ambiguous", 20), ("unknown", 20)]:
    pool = m2[m2["chk_strict"] == outcome]
    parts.append(pool.sample(min(n, len(pool)), random_state=42))
sample = pd.concat(parts).sample(frac=1, random_state=1)          # shuffled

sheet = sample[["query_id", "product_id", "query", "product_title", "product_color"]].copy()
sheet["annotator_A"] = ""
sheet["annotator_B"] = ""
sheet.to_csv("annotations/color_validation_v1.csv", index=False)                       # give this to annotators
sample[["query_id", "product_id", "chk_strict", "chk_lenient", "esci_label"]] \
      .to_csv("annotations/color_validation_v1_key.csv", index=False)                  # keep this hidden until done
print("Saved", len(sheet), "pairs. Fill annotator_A and annotator_B with: ok / violates / unclear")

Saved 200 pairs. Fill annotator_A and annotator_B with: ok / violates / unclear


evaluate the checker (run only after labeling)

In [20]:
from sklearn.metrics import cohen_kappa_score

ANN    = "../annotations/"                        # annotations/ is at the repo root
FILLED = ANN + "color_validation_v1_filled_AI.csv"  # <- your renamed file
KEY    = ANN + "color_validation_v1_key.csv"

ann = pd.read_csv(FILLED)
done = ann.dropna(subset=["annotator_A", "annotator_B"])

if len(done) < 20:
    print(f"Fill annotator_A and annotator_B in {FILLED} first.")
else:
    print("Pairs labeled by both:", len(done))
    print("Cohen's kappa (annotator agreement):", round(cohen_kappa_score(done["annotator_A"], done["annotator_B"]), 3))
    print("Note: annotator labels are AI-generated, so kappa is agreement between AI runs, not people.")

    agreed = done[done["annotator_A"] == done["annotator_B"]].merge(
        pd.read_csv(KEY), on=["query_id", "product_id"])
    agreed["annot_violates"] = agreed["annotator_A"] == "violates"

    flagged = agreed[agreed["chk_strict"] == "mismatch"]
    passed  = agreed[agreed["chk_strict"] == "match"]
    print(f"\nWhen checker says MISMATCH: annotators agree it violates {flagged['annot_violates'].mean():.1%} of the time (precision)")
    print(f"When checker says MATCH:    annotators say it violates   {passed['annot_violates'].mean():.1%} of the time (missed violations)")
    print("\nAnnotator labels by checker outcome:")
    print(pd.crosstab(agreed["chk_strict"], agreed["annotator_A"]))
    print("\nNote: the sample is stratified, so do not read these as overall rates.")

Pairs labeled by both: 200
Cohen's kappa (annotator agreement): 0.959
Note: annotator labels are AI-generated, so kappa is agreement between AI runs, not people.

When checker says MISMATCH: annotators agree it violates 58.2% of the time (precision)
When checker says MATCH:    annotators say it violates   3.9% of the time (missed violations)

Annotator labels by checker outcome:
annotator_A  not_color_request  ok  unclear  violates
chk_strict                                           
ambiguous                    3   2       11         0
match                       12  56        5         3
mismatch                    12   9       12        46
unknown                      5   1       10         4

Note: the sample is stratified, so do not read these as overall rates.


In [21]:
def chk_now(q, c, t):
    out = color_check(query_colors(q, False), "" if pd.isna(c) else str(c), "" if pd.isna(t) else str(t), False)
    return out or "no_color_request"

agreed["chk_v21"] = [chk_now(q, c, t) for q, c, t in zip(agreed["query"], agreed["product_color"], agreed["product_title"])]

f = agreed[agreed["chk_v21"] == "mismatch"]
p = agreed[agreed["chk_v21"] == "match"]
decided = f[f["annotator_A"].isin(["ok", "violates"])]
print(f"v2.1 MISMATCH precision: {f['annot_violates'].mean():.1%} of {len(f)}  "
      f"(clear-cut only: {decided['annot_violates'].mean():.1%} of {len(decided)})")
print(f"v2.1 MATCH but violates: {p['annot_violates'].mean():.1%} of {len(p)}")
print()
print(pd.crosstab(agreed["chk_v21"], agreed["annotator_A"]))

v2.1 MISMATCH precision: 66.7% of 69  (clear-cut only: 83.6% of 55)
v2.1 MATCH but violates: 3.1% of 64

annotator_A       not_color_request  ok  unclear  violates
chk_v21                                                   
ambiguous                         0   2       11         0
match                             1  56        5         2
mismatch                          2   9       12        46
no_color_request                 28   0        0         1
unknown                           1   1       10         4


In [22]:
cols = ["query", "product_color", "product_title"]
print(agreed[(agreed["chk_v21"] == "mismatch") & (agreed["annotator_A"] == "ok")][cols].to_string())


                                  query      product_color                                                                                                                                                                            product_title
17                  turquoise vinyl 651               Blue            (10) 12" x 12" Sheets - Oracal 651 Turquoise Adhesive Craft Vinyl for Cricut, Silhouette, Cameo, Craft Cutters, Printers, and Decals - Gloss Finish and Outdoor and Permanent
27                silver tablecloth 120               Grey                                                                                        My Jolie Home Silver Circle Rectangular Stain Resistant Tablecloth, 60 x 120-Inch - Grey & Silver
56                     tan onesie adult              Beige                                                                                                             Leveret Womens Pajamas Solid 2 Piece Pajama Set 100% Cotton Beige Size Large
58                  car 

In [23]:
def chk_len(q, c, t):
    out = color_check(query_colors(q, True), "" if pd.isna(c) else str(c), "" if pd.isna(t) else str(t), True)
    return out or "no_color_request"

agreed["chk_v21_len"] = [chk_len(q, c, t) for q, c, t in zip(agreed["query"], agreed["product_color"], agreed["product_title"])]

f = agreed[agreed["chk_v21_len"] == "mismatch"]
p = agreed[agreed["chk_v21_len"] == "match"]
decided = f[f["annotator_A"].isin(["ok", "violates"])]
print(f"v2.1 lenient MISMATCH precision: {f['annot_violates'].mean():.1%} of {len(f)}  "
      f"(clear-cut only: {decided['annot_violates'].mean():.1%} of {len(decided)})")
print(f"v2.1 lenient MATCH but violates: {p['annot_violates'].mean():.1%} of {len(p)}")
print()
print(pd.crosstab(agreed["chk_v21_len"], agreed["annotator_A"]))

v2.1 lenient MISMATCH precision: 74.2% of 62  (clear-cut only: 93.9% of 49)
v2.1 lenient MATCH but violates: 2.8% of 71

annotator_A       not_color_request  ok  unclear  violates
chk_v21_len                                               
ambiguous                         0   2       11         0
match                             1  62        6         2
mismatch                          2   3       11        46
no_color_request                 28   0        0         1
unknown                           1   1       10         4
